In [ ]:
# Generador de datos mock - VisionGuard
# Archivo separado del pipeline principal (etl_visionguard.ipynb)
#
# Este notebook contiene el grafo con nodos de acceso (ticket 003.4) y el
# generador de rutas dirigidas (mock v5), separado del pipeline principal
# porque ya no hace falta correrlo cada vez -- el pipeline real ahora
# consulta la BD directo (extraer_eventos_detectados /
# extraer_recorridos_coordenadas).
#
# Usa este archivo cuando necesites generar MAS datos de prueba (otro
# periodo simulado, otra institucion, mas volumen para probar el pipeline
# antes de tener datos reales de dispositivos IoT). El resultado se
# inserta en cangurera_local via el .sql que este notebook genera al
# final (mismo patron que reemplazo_mock_v5.sql, tickets 003.1/003.2/003.4).

# 1. Importamos las librerías necesarias
import math
import random
from datetime import datetime, timedelta
import networkx as nx
from generador_rutas import cargar_red_caminos

print("Librerias importadas.")

In [ ]:
# ============================================================
# Grafo con nodos de acceso (ticket 003.4)
# Cada punto de interes/riesgo se agrega como su propio nodo, conectado
# al andador real mas cercano por una arista de "acceso" (simula caminar
# del edificio/estacionamiento a la banqueta). Esto garantiza que el
# grafo quede en 1 solo componente conectado, y que las rutas que deben
# cruzar un punto de riesgo lo hagan exacto (distancia 0), sin depender
# de umbrales de cercania inestables.
# ============================================================

# 2. Puntos de interés y puntos de riesgo (coordenadas reales)
PUNTOS_INTERES = {
    "Entrada A (autos/motos)": (21.063707015763327, -101.58608800705031),
    "Canchas Americano": (21.063814358104942, -101.585052751258),
    "Canchas Basquet": (21.064720801456563, -101.58408779061212),
    "Canchas Futbol": (21.06337902479519, -101.58378104815513),
    "Edificio B Pesado": (21.064325327866044, -101.58327763915618),
    "Edificio B": (21.064188168664696, -101.58254912582085),
    "Edificio A": (21.063347320362, -101.58254273535302),
    "Edificio C": (21.063955594077722, -101.58179505061413),
    "Edificio A Pesado": (21.06287620469662, -101.58212096447465),
    "Entrada A Principal (peatonal)": (21.06271344521116, -101.58172754292714),
    "Biblioteca": (21.063031255581514, -101.58123907989197),
    "Cafeteria": (21.062828496692823, -101.58073423293152),
    "Edificio D": (21.063633567121133, -101.58048500468523),
    "Edificio CVD": (21.062673445591482, -101.58028690018176),
    "Cajeros ATM": (21.06302543762953, -101.58052265942663),
    "Edificio E / SITO": (21.063383101232517, -101.57965424386425),
    "Edificio F": (21.06321016024924, -101.5791493968933),
    "Edificio Rectoria": (21.062693442356547, -101.57857565396391),
    "Estacionamiento": (21.062162248378776, -101.57865327544415),
    "Entrada C (estacionamiento)": (21.061751779006062, -101.57839453717669),
}

# Puntos de riesgo. Verificados por point-in-polygon contra el geojson real:
# "Estacionamiento A" y "Estacionamiento B" SI caen dentro de poligonos
# reales etiquetados amenity=parking (way/224754458 y way/224754460).
PUNTOS_DE_RIESGO = {
    "Estacionamiento A (sin pavimentar)": (21.062353, -101.579416),
    "Estacionamiento B": (21.062636, -101.578256),
    "Acera con ramas bajas": (21.064192, -101.583074),
    "Pasillo con extintores colgados": (21.063279, -101.579530),
}

PESOS_RIESGO = {
    "Estacionamiento A (sin pavimentar)": 0.25,
    "Estacionamiento B": 0.20,
    "Acera con ramas bajas": 0.20,
    "Pasillo con extintores colgados": 0.35,
}

print(f"Puntos de interés: {len(PUNTOS_INTERES)} | Puntos de riesgo: {len(PUNTOS_DE_RIESGO)}")

In [ ]:
# 3. Funciones del grafo (base + nodos de acceso)
def _distancia_m(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def construir_grafo_base(caminos):
    # Grafo solo con los nodos/aristas de los andadores reales (LineStrings)
    G = nx.Graph()
    for camino in caminos:
        nodos_camino = [(round(lat, 7), round(lon, 7)) for lon, lat in camino]
        for i in range(len(nodos_camino) - 1):
            n1, n2 = nodos_camino[i], nodos_camino[i + 1]
            G.add_edge(n1, n2, weight=_distancia_m(n1[0], n1[1], n2[0], n2[1]))
    print(f"Grafo base: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
    return G

def conectar_componentes_cercanos(G, distancia_max_m=8):
    componentes = list(nx.connected_components(G))
    print(f"Componentes antes de conectar: {len(componentes)}")
    conexiones = 0
    for i in range(len(componentes)):
        for j in range(i + 1, len(componentes)):
            mejor_par = None
            mejor_dist = float("inf")
            for n1 in componentes[i]:
                for n2 in componentes[j]:
                    d = _distancia_m(n1[0], n1[1], n2[0], n2[1])
                    if d < mejor_dist:
                        mejor_dist = d
                        mejor_par = (n1, n2)
            if mejor_par and mejor_dist <= distancia_max_m:
                G.add_edge(mejor_par[0], mejor_par[1], weight=mejor_dist)
                conexiones += 1
    print(f"Conexiones agregadas: {conexiones}")
    print(f"Componentes después de conectar: {nx.number_connected_components(G)}")
    return G

def es_nodo_camino(n):
    # True solo si es un nodo de andador real (lat, lon), no un POI ya agregado
    return isinstance(n, tuple) and len(n) == 2 and n[0] != "POI"

def nodo_mas_cercano(G, lat, lon):
    # Busca solo entre nodos de camino real -- FIX ticket 003.4: antes
    # iteraba TODOS los nodos, incluyendo POIs ya agregados en una llamada
    # previa, lo que causaba TypeError al intentar math.radians() sobre un
    # string ('POI', nombre).
    mejor_nodo, mejor_dist = None, float("inf")
    for nodo in G.nodes:
        if not es_nodo_camino(nodo):
            continue
        d = _distancia_m(lat, lon, nodo[0], nodo[1])
        if d < mejor_dist:
            mejor_dist, mejor_nodo = d, nodo
    return mejor_nodo, mejor_dist

def agregar_nodos_de_acceso(G, puntos):
    # Agrega cada punto (POI o riesgo) como nodo propio, con una arista
    # de acceso al nodo de camino real mas cercano.
    for nombre, (lat, lon) in puntos.items():
        nodo_punto = ("POI", nombre)
        nodo_cercano, dist = nodo_mas_cercano(G, lat, lon)
        G.add_node(nodo_punto, lat=lat, lon=lon)
        G.add_edge(nodo_punto, nodo_cercano, weight=dist, tipo="acceso")
    return G

def coords_de_nodo(G, n):
    if isinstance(n, tuple) and n[0] == "POI":
        d = G.nodes[n]
        return d["lat"], d["lon"]
    return n[0], n[1]

def generar_ruta_con_nombre(G, nombre_origen, nombre_destino):
    # Ruta mas corta (Dijkstra) entre 2 puntos con nombre (POI o riesgo)
    no, nd = ("POI", nombre_origen), ("POI", nombre_destino)
    camino_nodos = nx.shortest_path(G, no, nd, weight="weight")
    return [coords_de_nodo(G, n) for n in camino_nodos]

print("Funciones del grafo definidas.")

In [ ]:
# 4. Construcción del grafo ampliado
CAMINOS = cargar_red_caminos("mapa_utl.geojson")
GRAFO_CAMPUS = construir_grafo_base(CAMINOS)
GRAFO_CAMPUS = conectar_componentes_cercanos(GRAFO_CAMPUS, distancia_max_m=8)
GRAFO_CAMPUS = agregar_nodos_de_acceso(GRAFO_CAMPUS, PUNTOS_INTERES)
GRAFO_CAMPUS = agregar_nodos_de_acceso(GRAFO_CAMPUS, PUNTOS_DE_RIESGO)
print(f"Grafo ampliado: {GRAFO_CAMPUS.number_of_nodes()} nodos, "
      f"{nx.number_connected_components(GRAFO_CAMPUS)} componente(s)")

In [ ]:
# ============================================================
# Generacion de mock (rutas dirigidas, grafo ampliado)
# Cambia n_recorridos, PARES_RUTA, o el seed si necesitas un volumen o
# distribucion distinta la proxima vez que generes datos de prueba.
# ============================================================

# 5. Generación de mock de datos simulados
random.seed(42)  # reproducibilidad, estilo León

DISTRIBUCION_EVENTOS = {
    2: 0.50,  # Obstaculo (Baja)
    5: 0.35,  # Tropiezo (Media)
    4: 0.15,  # Caida_Detectada (Critica)
}

# 5 pares de ruta dirigidos. 2 de ellos terminan EXACTO en el punto de
# riesgo (distancia 0 garantizada por el nodo de acceso); los otros pasan
# lo suficientemente cerca (verificado numéricamente).
PARES_RUTA = [
    ("Entrada C (estacionamiento)", "Estacionamiento A (sin pavimentar)"),
    ("Estacionamiento", "Estacionamiento B"),
    ("Canchas Basquet", "Edificio B"),
    ("Canchas Basquet", "Edificio E / SITO"),
    ("Entrada A (autos/motos)", "Edificio Rectoria"),
]

def elegir_tipo_evento():
    r = random.random()
    acumulado = 0
    for tipo_id, prob in DISTRIBUCION_EVENTOS.items():
        acumulado += prob
        if r <= acumulado:
            return tipo_id
    return list(DISTRIBUCION_EVENTOS.keys())[-1]

def elegir_punto_riesgo():
    nombres = list(PUNTOS_DE_RIESGO.keys())
    pesos = [PESOS_RIESGO[n] for n in nombres]
    nombre = random.choices(nombres, weights=pesos, k=1)[0]
    lat, lon = PUNTOS_DE_RIESGO[nombre]
    return nombre, lat, lon

def generar_recorrido_dirigido(origen, destino, seed, jitter_gps=0.00002, intervalo_s=30):
    rng = random.Random(seed)
    nodos_ruta = generar_ruta_con_nombre(GRAFO_CAMPUS, origen, destino)
    fecha_inicio = datetime.now()
    recorrido = []
    for i, (lat, lon) in enumerate(nodos_ruta):
        lat_r = lat + rng.uniform(-jitter_gps, jitter_gps)
        lon_r = lon + rng.uniform(-jitter_gps, jitter_gps)
        recorrido.append({
            "lat": round(lat_r, 6),
            "lon": round(lon_r, 6),
            "timestamp": fecha_inicio + timedelta(seconds=i * intervalo_s),
        })
    return recorrido

def generar_mock_completo(n_recorridos=15):
    # FIX (ticket 003.4): grafo ampliado con nodos de acceso -- elimina
    # NetworkXNoPath y garantiza que las rutas que deben cruzar un punto de
    # riesgo lo hagan de verdad.
    # FIX (ticket 003.2, se mantiene): timestamp de evento anclado a un punto
    # real de SU PROPIO recorrido, jitter +-15s, clamp a la ventana.
    recorridos = []
    eventos = []

    for i in range(1, n_recorridos + 1):
        origen, destino = PARES_RUTA[(i - 1) % len(PARES_RUTA)]
        ruta = generar_recorrido_dirigido(origen, destino, seed=i)
        recorridos.append({"recorrido_id": i, "coordenadas": ruta})

        n_eventos = random.randint(2, 4)
        for _ in range(n_eventos):
            nombre_riesgo, lat_r, lon_r = elegir_punto_riesgo()
            lat_jitter = lat_r + random.uniform(-0.00005, 0.00005)
            lon_jitter = lon_r + random.uniform(-0.00005, 0.00005)

            punto_base = random.choice(ruta)
            timestamp_evento = punto_base["timestamp"] + timedelta(seconds=random.randint(-15, 15))
            ts_min, ts_max = ruta[0]["timestamp"], ruta[-1]["timestamp"]
            timestamp_evento = max(ts_min, min(ts_max, timestamp_evento))

            eventos.append({
                "recorrido_id": i,
                "tipo_evento_id": elegir_tipo_evento(),
                "latitud": lat_jitter,
                "longitud": lon_jitter,
                "geo_es_estimado": False,
                "timestamp": timestamp_evento,
            })

    print(f"Mock generado: {len(recorridos)} recorridos, {len(eventos)} eventos.")
    return recorridos, eventos

recorridos_mock, eventos_mock = generar_mock_completo()

In [ ]:
# ============================================================
# Exportar a .sql para insertar en cangurera_local
# Mismo patron que reemplazo_mock_v5.sql (tickets 003.1/003.2/003.4):
# tabla variable @NewId + OUTPUT INSERTED.Id, rotacion de DispositivoId
# entre 8 y 10, bloques separados por GO.
# ============================================================

# 6. Generar el script .sql de insercion
def generar_sql_insercion(recorridos, eventos, nombre_archivo="insercion_mock_nuevo.sql"):
    lines = []
    lines.append("USE [Cangurera_Local];")
    lines.append("GO")
    lines.append("")
    for rec in recorridos:
        rid = rec["recorrido_id"]
        coords = rec["coordenadas"]
        dispositivo_id = 8 if rid % 2 == 1 else 10
        fecha_inicio_r = coords[0]["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
        fecha_fin_r = coords[-1]["timestamp"].strftime("%Y-%m-%d %H:%M:%S")

        lines.append(f"-- Recorrido mock #{rid} (DispositivoId {dispositivo_id})")
        lines.append("DECLARE @NewId TABLE (Id BIGINT);")
        lines.append("INSERT INTO Operativo.Recorridos (DispositivoId, FechaInicio, FechaFin, Ruta_Coordenadas)")
        lines.append("OUTPUT INSERTED.Id INTO @NewId")
        lines.append(f"VALUES ({dispositivo_id}, '{fecha_inicio_r}', '{fecha_fin_r}', NULL);")
        lines.append("")
        lines.append("DECLARE @RecId BIGINT = (SELECT TOP 1 Id FROM @NewId);")
        lines.append("")
        for c in coords:
            fecha = c["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
            lines.append(
                f"INSERT INTO Operativo.RecorridoCoordenadas (RecorridoId, Fecha, Latitud, Longitud) "
                f"VALUES (@RecId, '{fecha}', {c['lat']}, {c['lon']});"
            )
        lines.append("")
        eventos_rec = [e for e in eventos if e["recorrido_id"] == rid]
        for e in eventos_rec:
            ts = e["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
            geo_est = 0 if not e["geo_es_estimado"] else 1
            lines.append(
                f"INSERT INTO Operativo.Eventos_Detectados "
                f"(RecorridoId, TipoEventoId, TimestampEvento, Latitud, Longitud, Geo_Es_Estimado) "
                f"VALUES (@RecId, {e['tipo_evento_id']}, '{ts}', {round(e['latitud'],6)}, {round(e['longitud'],6)}, {geo_est});"
            )
        lines.append("GO")
        lines.append("")

    lines.append("-- Verificacion")
    lines.append("SELECT COUNT(*) AS TotalRecorridos FROM Operativo.Recorridos;")
    lines.append("SELECT COUNT(*) AS TotalCoordenadas FROM Operativo.RecorridoCoordenadas;")
    lines.append("SELECT COUNT(*) AS TotalEventos FROM Operativo.Eventos_Detectados;")

    with open(nombre_archivo, "w") as f:
        f.write("\n".join(lines))
    print(f"Script generado: {nombre_archivo}")

generar_sql_insercion(recorridos_mock, eventos_mock)